# SHAPE — Self-Healing Autonomous Programming Engine
## Notebook 4 of 4: Deep Code Walkthrough & Technical Summary

**Group 3 | ITAI2376-14562**  
Cherluk Sumdin · Olubukola Kukoyi · Taki Eddine Boubekri · Michael Rivas

---

This final notebook dissects the most technically significant parts of the codebase:
the safety regex engine, the RL policy math, the API server endpoints, and the
complete ReAct trace with timing. It closes with a technical summary table of
every component and its contribution to the system.


## 1. InputGuard — Regex Safety Engine (shape/safety.py)

`InputGuard.check()` runs in O(P) time where P is the number of compiled patterns
(currently 11 blocked + 3 out-of-scope). Every call completes in < 1 ms regardless
of input length (up to the 8,000-char limit). This is **pre-LLM** — the check
runs before `llm.invoke()` is ever called, burning zero Ollama compute.

Pattern categories:
1. **Security/harmful** (`_BLOCKED_PATTERNS`): OS system calls, destructive shell
   commands, `eval(input())`, dynamic `__import__('os')`, malware keywords
2. **Out-of-scope** (`_OUT_OF_SCOPE_PATTERNS`): medical advice, legal advice,
   non-coding creative writing
3. **Length guard**: rejects inputs > 8,000 chars (prevents prompt injection
   via absurdly long inputs)
4. **User-defined extensions** (`extra_blocked_patterns` in config): regex patterns
   can be added at runtime in `config.yaml` without code changes


In [ ]:
# Inspect every regex pattern in InputGuard and show its match groups.
# This proves we wrote these patterns, not borrowed a library.

import sys, os
sys.path.insert(0, os.path.abspath(".."))

from shape.safety import _BLOCKED_PATTERNS, _OUT_OF_SCOPE_PATTERNS, InputGuard
import re, time

print("_BLOCKED_PATTERNS (security/harmful):")
print(f"  {'Pattern (abbreviated)':<55} {'Description'}")
print("-" * 95)
for pat, desc in _BLOCKED_PATTERNS:
    print(f"  {pat[:53]:<55} {desc}")

print()
print("_OUT_OF_SCOPE_PATTERNS (soft-block):")
for pat, desc in _OUT_OF_SCOPE_PATTERNS:
    print(f"  {pat[:53]:<55} {desc}")

print()
print("Timing (average over 1000 calls):")
guard = InputGuard({})
sample = "Write a Python function to parse sensor CSV data and plot it."
N = 1000
t0 = time.perf_counter()
for _ in range(N):
    guard.check(sample)
avg_us = (time.perf_counter() - t0) / N * 1e6
print(f"  Input: {repr(sample[:45])}")
print(f"  Average check time: {avg_us:.1f} µs per call (pre-LLM)")

In [ ]:
# Demonstrate the runtime-extensible extra_blocked_patterns from config.
# A site admin can add patterns in config.yaml without touching Python.

extended_guard = InputGuard({
    "extra_blocked_patterns": [
        r"(?i)\bwrite.*bitcoin.*wallet",   # crypto abuse
        r"(?i)\bgenerate.*password.*dump",  # credential harvesting
    ]
})

extended_cases = [
    "Write a bitcoin wallet generator in Python",
    "Generate a password dump script",
    "Explain how hashing works in Python",
]
print("Custom extra_blocked_patterns in action:")
for text in extended_cases:
    blocked, reason = extended_guard.check(text)
    print(f"  {text[:55]:<55} {'BLOCKED: ' + reason[:30] if blocked else 'ALLOWED'}")

## 2. The RL Feedback Loop — Mathematics & Policy (shape/feedback.py)

`FeedbackMemory` implements a simplified **multi-armed bandit** reward loop:

**State:** A circular buffer of the last 20 `(input, output, score)` triples,
persisted as JSONL on disk so learning survives Pi reboots.

**Reward signal:** Rolling average of the last `N` scores (N = min(20, records)):

$$\bar{r}_t = \frac{1}{N} \sum_{i=t-N+1}^{t} r_i, \quad r_i \in \{1, 2, 3, 4, 5\}$$

**Policy update (prompt injection):**

| Average score | Policy injected into next system prompt |
|---|---|
| $\bar{r} < 2.5$ | Correction hint: be more thorough, test code, explain decisions |
| $2.5 \le \bar{r} < 3.5$ | Nudge: double-check edge cases, always lint first |
| $\bar{r} \ge 3.5$ | Positive hint: keep the same careful approach |
| `records < 5` | No hint (insufficient data) |

This is not full RL (no Bellman equation, no policy gradient), but it is a
genuine online learning loop: feedback drives measurable prompt-strategy changes
between sessions.


In [ ]:
# Simulate the full RL learning curve.
# Shows how the policy hint evolves as the agent improves.

from pathlib import Path
from shape.feedback import FeedbackMemory, MIN_RECORDS_FOR_HINT

Path("../data").mkdir(exist_ok=True)
fm = FeedbackMemory(Path("../data/demo_rl_nb4.jsonl"))

# Score sequence: starts poor → gradually improves
score_sequence = [2, 1, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 5, 4, 5, 5, 4, 5, 5]

print(f"{'Turn':<6} {'Score':<8} {'Rolling avg':<13} {'Policy hint (truncated)'}")
print("-" * 85)

for i, score in enumerate(score_sequence, 1):
    fm.record({"input": f"task_{i}", "output": "response", "score": score})
    scores = [r["score"] for r in fm._buffer if "score" in r]
    avg    = sum(scores) / len(scores) if scores else 0
    hint   = fm.get_policy_hint()
    if not hint:
        hint_short = f"(waiting for {MIN_RECORDS_FOR_HINT} records)"
    else:
        # First 8 words
        hint_short = " ".join(hint.split()[:8]) + "..."
    print(f"  {i:<4}  {score:<6}  {avg:<13.2f}  {hint_short}")

print()
print(f"Final summary: {fm.summary()}")

In [ ]:
# Prove persistence: scores survive a session restart.
# This is critical on a Pi that reboots frequently in the field.

jsonl_path = Path("../data/demo_rl_nb4.jsonl")
print(f"JSONL file size: {jsonl_path.stat().st_size} bytes")
print(f"Contents (last 5 lines):")
lines = jsonl_path.read_text().splitlines()
import json
for line in lines[-5:]:
    rec = json.loads(line)
    print(f"  score={rec['score']}, input={rec['input']!r}")

# Reload into a new FeedbackMemory instance (simulates Pi reboot)
fm2 = FeedbackMemory(jsonl_path)
stats2 = fm2.summary()
print()
print("After simulated reboot (new FeedbackMemory() loading same JSONL):")
print(f"  Records loaded:  {stats2['records']}")
print(f"  Average score:   {stats2['avg_score']}")
print(f"  Policy hint:     {stats2['policy'][:70]}...")

## 3. Pi 5 HTTP API Server (hardware/pi5/api_server.py)

The API server wraps the SHAPE agent in a minimal `BaseHTTPRequestHandler`
(no framework dependency — avoids installing Flask/FastAPI on the Pi).

**Endpoints:**

| Method | Path | Body | Returns | Used by |
|---|---|---|---|---|
| POST | `/query` | `{"input": "..."}` | `{"output": "..."}` | Zero 2 W K1 button |
| POST | `/feedback` | `{"input","output","score"}` | `{"ok": true}` | Zero 2 W RATE state |
| GET  | `/status` | — | `{"model","rl","avg_reward"}` | Zero 2 W STATUS screen |
| POST | `/reset` | — | `{"ok": true}` | — |
| POST | `/sensor` | `{"aqi","temperature","air_alert",...}` | `{"cmd","text","reward"}` | SerialBridge auto-forward |
| POST | `/transcribe` | raw WAV bytes | `{"text": "..."}` | Zero 2 W K3 long-press |

**Authentication:** Every request must carry `X-API-Key: shape_local_key` (or the
configured value). Without this header the server returns 401 immediately — no
agent compute is wasted on unauthorized requests. This matters on the Pi's
shared local Wi-Fi network.

**Voice transcription path:** `/transcribe` saves the WAV to a tempfile,
calls `whisper.cpp main` as a subprocess with the `ggml-tiny.en` model,
captures stdout, unlinks the temp file, and returns the text. The Zero 2 W
then uses the transcript to pre-fill its QUERY state.


In [ ]:
# Reproduce the /sensor auto-query logic from api_server.py.
# Shows exactly how sensor telemetry becomes an agent query automatically.

def sensor_endpoint_logic(payload: dict, agent) -> dict:
    """
    Verbatim reproduction of the /sensor POST handler in api_server.py.
    Returns the dict that would be JSON-encoded back to the Zero 2 W.
    """
    aqi   = payload.get("aqi")
    temp  = payload.get("temperature")
    alert = payload.get("air_alert", False)

    if alert or (aqi and aqi > 600) or (temp and temp > 40):
        concern = []
        if alert or (aqi and aqi > 600):
            concern.append(f"AQI={aqi:.0f}" if aqi else "air alert")
        if temp and temp > 40:
            concern.append(f"temp={temp:.1f}°C")
        query = (
            "SHAPE ALERT from Pico 2 sensor: "
            + ", ".join(concern)
            + ". What action should be taken?"
        )
        output = agent.run(query)
        rl     = agent.feedback.summary()
        return {
            "cmd":    "ALERT",
            "text":   output[:120],
            "reward": rl.get("avg_score"),
        }
    else:
        rl = agent.feedback.summary()
        return {
            "cmd":    "STATUS",
            "text":   "Sensors nominal",
            "reward": rl.get("avg_score"),
        }


from shape.agent import build_agent
import sys, os
sys.path.insert(0, os.path.abspath(".."))

agent = build_agent({
    "model": "codellama", "ollama_url": "http://localhost:11434",
    "temperature": 0.2, "context_window": 4096,
    "memory_window": 6, "max_iterations": 8,
    "executor_timeout": 10, "feedback_path": "../data/feedback.jsonl",
    "workspace_dir": "../workspace", "doc_index_path": "../data/doc_index",
    "docs_dir": "../data/docs", "doc_top_k": 3, "verbose": False,
})

print("=" * 60)
print("CASE 1 — Normal sensor reading (AQI=312, temp=22°C)")
print("=" * 60)
r1 = sensor_endpoint_logic({"aqi": 312, "temperature": 22, "air_alert": False}, agent)
print(f"  Response to Zero 2 W: {r1}")

print()
print("=" * 60)
print("CASE 2 — Alert: AQI=648 (auto-query triggered)")
print("=" * 60)
r2 = sensor_endpoint_logic({"aqi": 648, "temperature": 24, "air_alert": True}, agent)
print(f"  cmd:    {r2['cmd']}")
print(f"  text:   {r2['text'][:100]}...")
print(f"  reward: {r2['reward']}")

In [ ]:
# Full ReAct trace with iteration counting and timing.
# This is the clearest proof that our agent uses a multi-step loop,
# not a single LLM call. verbose=True in the agent prints each step.

import time

# Temporarily re-enable verbose to capture the trace
agent.config["verbose"] = True

print("=" * 65)
print("FULL ReAct TRACE — multi-step code generation")
print("Tools expected: linter → code_executor (at minimum)")
print("=" * 65)

t_start = time.perf_counter()

result = agent.run(
    "Write a Python function that converts a temperature reading in Celsius "
    "to Fahrenheit and to Kelvin. Lint it first, then run it with 100°C and 0°C."
)

elapsed = time.perf_counter() - t_start

print("\n" + "=" * 65)
print("FINAL ANSWER")
print("=" * 65)
print(result)
print(f"\nTotal wall time: {elapsed:.1f} s  (includes Ollama LLM forward passes)")

agent.config["verbose"] = False

In [ ]:
# Generate a live codebase overview: line counts, component sizes.
# Shows the actual size of what we built.

from pathlib import Path

source_files = [
    ("shape/agent.py",             "Core ReAct agent loop"),
    ("shape/safety.py",            "InputGuard regex engine"),
    ("shape/feedback.py",          "RL feedback + policy hints"),
    ("shape/tools/__init__.py",    "Tool registry"),
    ("shape/tools/code_executor.py","Sandboxed subprocess executor"),
    ("shape/tools/linter.py",      "pyflakes static analysis"),
    ("shape/tools/file_manager.py","Workspace file I/O"),
    ("shape/tools/doc_search.py",  "Offline FAISS doc search"),
    ("main.py",                    "CLI entry point"),
    ("hardware/pi5/api_server.py", "Pi 5 HTTP API server"),
    ("hardware/zero2w/main_zero2w.py", "Zero 2 W field terminal"),
    ("hardware/zero2w/serial_bridge.py", "UART ↔ HTTP bridge"),
    ("hardware/pico2/main.py",     "Pico 2 sensor firmware"),
    ("hardware/pico2/sensors.py",  "DHT22 + MQ-135 driver"),
]

base = Path("..")
total_lines = 0
total_code  = 0

print(f"{'File':<42} {'Lines':<8} {'Code':<8} {'Description'}")
print("-" * 95)

for rel, desc in source_files:
    path = base / rel
    if not path.exists():
        print(f"  {rel:<42} {'(not found)':<8}")
        continue
    lines = path.read_text(encoding="utf-8").splitlines()
    total  = len(lines)
    code   = sum(1 for l in lines if l.strip() and not l.strip().startswith("#"))
    total_lines += total
    total_code  += code
    print(f"  {rel:<42} {total:<8} {code:<8} {desc}")

print("-" * 95)
print(f"  {'TOTAL':<42} {total_lines:<8} {total_code:<8} lines")

## 4. Course Concepts Map

Every major concept from ITAI2376 is implemented in production code in SHAPE:

| Course concept | SHAPE implementation | File |
|---|---|---|
| Agent architecture | `SHAPEAgent` class with `run()` loop | `shape/agent.py` |
| ReAct (Reason + Act) | Prompt-based Thought/Action/Observation cycle | `shape/agent.py:REACT_SYSTEM` |
| Tool use | 4 real tools: linter, executor, file manager, doc search | `shape/tools/` |
| Memory | `InMemoryChatMessageHistory` with rolling window | `shape/agent.py:run` |
| Reinforcement learning | Bandit-style reward loop, policy hint injection | `shape/feedback.py` |
| Safety / guardrails | Pre-LLM regex screening with 11 patterns | `shape/safety.py` |
| Offline LLM | CodeLlama 7B via Ollama on Raspberry Pi | `config.yaml` |
| RAG (retrieval) | FAISS + all-MiniLM-L6-v2 offline semantic search | `shape/tools/doc_search.py` |
| Multi-agent / multi-node | 3-tier Pi 5 / Zero 2 W / Pico 2 deployment | `hardware/` |
| API design | 6-endpoint HTTP server, API key auth | `hardware/pi5/api_server.py` |
| Embedded / edge AI | MicroPython firmware, ST7789 LCD, DHT22/MQ-135 | `hardware/pico2/` |
| Voice interface | whisper.cpp `/transcribe` endpoint | `api_server.py:do_POST /transcribe` |


In [ ]:
# FINAL DEMONSTRATION — full pipeline in one cell.
# Mimics what happens when the researcher presses K1 on the Zero 2 W
# and the query travels through Pi5Client → api_server → agent.run()
# → tools → back to Zero 2 W LCD → Pico 2 reward bar.

import json

# Step 1: Query arrives at the Pi 5 /query endpoint
incoming_request = {
    "X-API-Key": "shape_local_key",
    "input": (
        "I need to process a CSV file of sensor readings. "
        "Write a Python function that reads sensor_data.csv from the workspace, "
        "computes a 3-sample rolling average for co2_ppm, and flags any window "
        "where the average exceeds 420 ppm. Test it and show the output."
    )
}

print("=" * 70)
print("STEP 1: Request arrives at Pi 5 /query endpoint")
print(f"  Auth key: {incoming_request['X-API-Key']}  (matches config)")
print(f"  Input: {incoming_request['input'][:70]}...")

# Step 2: agent.run() (ReAct loop)
agent.config["verbose"] = True
print()
print("=" * 70)
print("STEP 2: agent.run() — ReAct loop executing")
print("=" * 70)

output = agent.run(incoming_request["input"])
agent.config["verbose"] = False

# Step 3: Response
response_body = {"output": output}
print()
print("=" * 70)
print("STEP 3: /query response body → Zero 2 W → paginated on LCD")
print("=" * 70)
print(output[:400], "..." if len(output) > 400 else "")

# Step 4: Rating (simulates user pressing RIGHT RIGHT PRESS in RATE state)
rating = 5
agent.collect_feedback(incoming_request["input"], output, rating)
stats  = agent.feedback.summary()

# Step 5: Reward mirrored back to Pico 2 LCD
pico_cmd = {
    "cmd":    "STATUS",
    "text":   output[:40],
    "reward": round(stats['avg_score'] / 5.0, 2) if stats['avg_score'] else None,
}
print()
print("=" * 70)
print("STEP 4-5: Feedback recorded; reward sent to Pico 2 LCD")
print("=" * 70)
print(f"  Rating submitted:  {rating}/5")
print(f"  RL avg score:      {stats['avg_score']}")
print(f"  Pico 2 JSON cmd:   {json.dumps(pico_cmd)}")
print(f"  Reward bar on LCD: {'█' * int((pico_cmd['reward'] or 0) * 20):<20} "
      f"{(pico_cmd['reward'] or 0)*100:.0f}%")

## Technical Summary

### What SHAPE actually is

SHAPE is a **production-grade offline AI coding agent** deployed across three
physical devices. Its architecture implements:

- A hand-rolled **prompt-based ReAct loop** (not a black-box API) because the
  local LLM has no tool-calling protocol
- **Four tools** that execute real actions: pyflakes linting, sandboxed subprocess
  execution, path-safe file I/O, and offline FAISS semantic search
- A **regex-based pre-LLM safety screen** with 11 compiled patterns and runtime
  extensibility via config
- An **online RL feedback loop** with rolling reward average, persistent JSONL
  history, and automatic policy-hint injection into every system prompt
- A **6-endpoint HTTP API** with key authentication, auto-alert sensor routing,
  and whisper.cpp voice transcription
- **MicroPython firmware** for a Pi Pico 2 with 240×240 ST7789 display, DHT22
  temperature/humidity, MQ-135 AQI sensing, and bidirectional UART communication

### Why this matters

> The most important insight was that intelligence does not have to be centralized.
> A $55 device running an open-source model, guided by structured reasoning and
> improved through feedback, can compete with cloud services on real coding tasks —
> and beats them entirely when there is no signal.
>
> — Group 3, ITAI2376

---

**Cherluk Sumdin · Olubukola Kukoyi · Taki Eddine Boubekri · Michael Rivas**  
Group 3 | ITAI2376-14562
